<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_4_Advanced_Multiclass_Extensions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: Predicting Credit Default: Part 5
## decision boundaries and Advanced metrics

author: Brad Sheese

---

## Introduction: How Models "See" the Data

We have reached the final notebook in our classification series! We have learned how to train models, evaluate them using complex metrics, and pick the best algorithm via cross-validation.

In this final installment, we will pull back the curtain to see how these models actually carve up the feature space. We will visualize decision boundaries—the geometric lines and shapes that separate one class from another. We will also learn how to calculate precision, recall, and f1-score when we have 3 or more classes using macro, weighted, and micro averaging.

### Learning objectives
By the end of this notebook, you will be able to:
1.  Visualize decision boundaries in 2D using pca.
2.  Compare the geometric logic of logistic regression (linear) vs. knn (non-linear) vs. decision trees (Axis-aligned).
3.  Interpret multiclass metrics (Macro vs. Weighted averaging).
4.  Understand feature importance in a multiclass context.

## problem 1: setup and Data Loading (wine dataset)

We will continue with the balanced wine dataset introduced in Part 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Setup complete. Classes: {np.unique(y)}")

## problem 2: Visualizing decision boundaries

It's hard to visualize 13 chemical features at once. To see the "decision boundaries," we use pca (principal component analysis) to squash the data into its two most important dimensions (pc1 and pc2).

We will then train three different models on these 2D features and plot their "territories."

In [ ]:
pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train_s)

def plot_boundaries(clf, X, y, ax, title):
    h = .02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='RdYlBu')
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

clfs = [
    ('Logistic Regression (Linear)', LogisticRegression()),
    ('Decision Tree (Axis-Aligned)', DecisionTreeClassifier(max_depth=3)),
    ('KNN (Local/Non-linear)', KNeighborsClassifier(n_neighbors=5))
]

for i, (name, clf) in enumerate(clfs):
    clf.fit(X_train_2d, y_train)
    plot_boundaries(clf, X_train_2d, y_train, axes[i], name)

plt.show()

### Look Closely:
- logistic regression: Notice how the boundaries are straight lines. This is why it's called a "linear" model.
- decision tree: Notice how the boundaries are composed of horizontal and vertical steps. This is because trees split on one variable at a time.
- knn: Notice the more complex, wavy boundaries. KNN looks at the neighbors around every single point, allowing it to adapt to very irregular shapes.

## problem 3: Multiclass confusion matrix

When we have 3 classes, our confusion matrix becomes a 3x3 grid. Let's see where the model is mixing up these wines.

In [ ]:
# Train a Random Forest on ALL 13 features
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(3), wine.target_names)
plt.yticks(range(3), wine.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('3x3 Multiclass Confusion Matrix')

for i in range(3):
    for j in range(3):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontweight='bold')
plt.show()

## problem 4: Averaging Strategies (Macro vs. Weighted)

With 3 classes, you have 3 different precision scores (one for each wine). How do you get a single number? 

- macro averaging: Calculate the metric for each class separately, then take the simple average. This treats every class as equally important, regardless of how many samples it has.
- weighted averaging: Calculate the metric for each class, but give more "weight" to the classes that have more samples. This is better if you want to reflect overall performance on the dataset population.

In [ ]:
print("Full Classification Report:")
print(classification_report(y_test, y_pred, target_names=wine.target_names))

## problem 5: feature importance

Finally, let's look at which features are most important for distinguishing these three cultivars. For tree-based models like random forest, we can use `feature_importances_` to see which variables led to the most "purity" when splitting the data.

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(X.shape[1]), importances[indices], color='darkgreen', edgecolor='black')
plt.xticks(range(X.shape[1]), [wine.feature_names[i] for i in indices], rotation=45, ha='right')
plt.title('What Drives Wine Classification? (Feature Importance)')
plt.ylabel('Relative Importance')
plt.tight_layout()
plt.show()

### Discussion:
Which chemical property is the biggest differentiator? (Often it's `proline` or `color_intensity`). Understanding these features allows you to explain why the model chose one cultivar over another.

## Summary of Geometric Model Behaviors

As we saw in the decision boundary plots:
- logistic regression creates linear boundaries. It assumes the classes can be separated by straight lines (or hyperplanes in higher dimensions).
- decision trees create axis-aligned rectangular boundaries. They divide the space based on one feature at a time, creating a "staircase" effect.
- k-nearest neighbors (knn) creates highly non-linear, local boundaries. It is very flexible and can capture extremely complex shapes, but it is sensitive to noise and the choice of K.
- random forests and gradient boosting create complex, non-linear boundaries by combining many trees, allowing them to capture intricate patterns while being more robust than a single tree.


## conclusion
Congratulations! You have completed the comprehensive Introduction to Classification series. 

Over these five notebooks, we have progressed from the mathematical foundations of the sigmoid function to the geometric intuition of decision boundaries. You now possess the toolkit required to train, tune, and evaluate classification models for both binary and multiclass real-world problems. 

Keep this toolkit handy—it is the foundation of almost every modern predictive system, from fraud detection to facial recognition.